In [2]:
import pandas as pd
import glob
import os

modis_dir = '/Users/anora/Team MG Dropbox/Wanru Wu/Energy_Flux/intermediate/modis'

all_files = glob.glob(os.path.join(modis_dir, '*.csv'))

files_1km = [f for f in all_files if os.path.basename(f).startswith('1km')]
files_5km = [f for f in all_files if os.path.basename(f).startswith('5km')]

print('1km files:', [os.path.basename(f) for f in sorted(files_1km)])
print('5km files:', [os.path.basename(f) for f in sorted(files_5km)])

1km files: ['1km_2020_1-101.csv', '1km_2020_101-200.csv', '1km_2020_201-300.csv', '1km_2020_301-366.csv']
5km files: ['5km_2020_1-101.csv', '5km_2020_101-200.csv', '5km_2020_201-300.csv', '5km_2020_301-366.csv']


In [3]:
df_1km = pd.concat([pd.read_csv(f, index_col=0) for f in sorted(files_1km)], ignore_index=True)
df_5km = pd.concat([pd.read_csv(f, index_col=0) for f in sorted(files_5km)], ignore_index=True)

print(f'df_1km shape: {df_1km.shape}')
print(f'df_5km shape: {df_5km.shape}')

df_1km shape: (72049484, 14)
df_5km shape: (2874521, 16)


In [5]:
import pandas as pd
import numpy as np

df = df_1km

# Convert qa1_byte columns to integer (handle NaN)
byte_cols = [col for col in df.columns if col.startswith('qa1_byte_')]
for col in byte_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Helper to extract bits from a byte column
def extract_bits(df, column, start, end):
    def _extract(x):
        if pd.isna(x):
            return pd.NA
        bin_str = format(int(x), '08b')
        return int(bin_str[-(end+1):8-start], 2)
    
    return df[column].apply(_extract)

# Extract all flags
df['usefulness_flag'] = extract_bits(df, 'qa1_byte_0', 0, 0)           # Byte 0, Bit 0
df['confidence_flag'] = extract_bits(df, 'qa1_byte_0', 1, 2)           # Byte 0, Bits 1-2
df['cloud_phase_flag'] = extract_bits(df, 'qa1_byte_2', 0, 2)          # Byte 2, Bits 0-2
df['cloud_retrieval_outcome'] = extract_bits(df, 'qa1_byte_2', 3, 3)   # Byte 2, Bit 3
df['16_21_usefulness'] = extract_bits(df, 'qa1_byte_3', 0, 0)          # Byte 3, Bit 0
df['16_21_confidence'] = extract_bits(df, 'qa1_byte_3', 1, 2)          # Byte 3, Bits 1-2
df['multilayer_phase_flag'] = extract_bits(df, 'qa1_byte_4', 3, 5)     # Byte 4, Bits 0-2

# Apply base mask — Usefulness Flag == 1
base_mask = df['usefulness_flag'] == 1
df_filtered = df[base_mask]
total_useful = len(df_filtered)

print(f"Total rows: {len(df)}")
print(f"Rows with Usefulness Flag == 1: {total_useful} ({total_useful/len(df)*100:.2f}%)\n")

# Calculate coverage of each mask within the filtered data
masks = {
    'Cloud Optical Thickness Confidence Flag == 3': df_filtered['confidence_flag'] == 3,
    'Cloud Retrieval Phase Flag != 0': df_filtered['cloud_phase_flag'] != 0,
    'Cloud Retrieval Outcome Flag == 1': df_filtered['cloud_retrieval_outcome'] == 1,
    '1.6-2.1µm Usefulness Flag == 1': df_filtered['16_21_usefulness'] == 1,
    '1.6-2.1µm Confidence Flag == 3': df_filtered['16_21_confidence'] == 3,
    'Multilayer Cloud & Phase Flag != 0': df_filtered['multilayer_phase_flag'] != 0
}

print(f"{'Mask':<55} {'Count':>8} {'Coverage':>10}")
print("-" * 75)
for name, mask in masks.items():
    count = mask.sum()
    pct = count / total_useful * 100 if total_useful > 0 else 0
    print(f"{name:<55} {count:>8} {pct:>9.2f}%")


Total rows: 72049484
Rows with Usefulness Flag == 1: 25434953 (35.30%)

Mask                                                       Count   Coverage
---------------------------------------------------------------------------
Cloud Optical Thickness Confidence Flag == 3            25434953    100.00%
Cloud Retrieval Phase Flag != 0                         25434953    100.00%
Cloud Retrieval Outcome Flag == 1                       23485251     92.33%
1.6-2.1µm Usefulness Flag == 1                               129      0.00%
1.6-2.1µm Confidence Flag == 3                               129      0.00%
Multilayer Cloud & Phase Flag != 0                      25434953    100.00%


In [ ]:
import pandas as pd
import numpy as np

df = df_5km

# Convert qa1_byte columns to integer (handle NaN)
byte_cols = [col for col in df.columns if col.startswith('qa1_byte_')]
for col in byte_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Helper to extract bits from a byte column
def extract_bits(df, column, start, end):
    def _extract(x):
        if pd.isna(x):
            return pd.NA
        bin_str = format(int(x), '08b')
        return int(bin_str[-(end+1):8-start], 2)
    
    return df[column].apply(_extract)

# Extract all flags
df['usefulness_flag'] = extract_bits(df, 'qa1_byte_0', 0, 0)           # Byte 0, Bit 0
df['confidence_flag'] = extract_bits(df, 'qa1_byte_0', 1, 2)           # Byte 0, Bits 1-2

# Apply base mask — Usefulness Flag == 1
base_mask = df['usefulness_flag'] == 1
df_filtered = df[base_mask]
total_useful = len(df_filtered)

print(f"Total rows: {len(df)}")
print(f"Rows with Usefulness Flag == 1: {total_useful} ({total_useful/len(df)*100:.2f}%)\n")

# Calculate coverage of each mask within the filtered data
masks = {
    'Cloud Optical Thickness Confidence Flag == 3': df_filtered['confidence_flag'] == 3,
    'Cloud Retrieval Phase Flag != 0': df_filtered['cloud_phase_flag'] != 0,
}

print(f"{'Mask':<55} {'Count':>8} {'Coverage':>10}")
print("-" * 75)
for name, mask in masks.items():
    count = mask.sum()
    pct = count / total_useful * 100 if total_useful > 0 else 0
    print(f"{name:<55} {count:>8} {pct:>9.2f}%")
